In [0]:
# Downloads the CMS QHP Public Use Files (public data only) into a Unity Catalog Volume
# and records a SHA-256 fingerprint of each original file, so later verification can
# prove the loaded data came from exactly these files.

import hashlib
import os
import zipfile

import pandas as pd
import requests

VOLUME_DIR = "/Volumes/<your_catalog>/<your_schema>"

# (plan_year, dataset, official CMS download URL)
CMS_SOURCE_FILES = [
    (2026, "plan_attributes",           "https://download.cms.gov/marketplace-puf/2026/plan-attributes-puf.zip"),
    (2026, "rates",                     "https://download.cms.gov/marketplace-puf/2026/rate-puf.zip"),
    (2026, "benefits_and_cost_sharing", "https://download.cms.gov/marketplace-puf/2026/benefits-and-cost-sharing-puf.zip"),
    (2027, "plan_attributes",           "https://download.cms.gov/marketplace-puf/2027/plan-attributes-puf.zip"),
    (2027, "rates",                     "https://download.cms.gov/marketplace-puf/2027/rate-puf.zip"),
    (2027, "benefits_and_cost_sharing", "https://download.cms.gov/marketplace-puf/2027/benefits-and-cost-sharing-puf.zip"),
]

# Some public sites reject requests that don't identify themselves.
REQUEST_HEADERS = {"User-Agent": "qhp-puf-mcp-hackathon/0.1 (public data research)"}
CHUNK_BYTES = 8 * 1024 * 1024  # stream in 8 MB pieces so large files don't exhaust memory


def download_and_fingerprint(plan_year: int, dataset: str, source_url: str) -> dict:
    year_folder = f"{VOLUME_DIR}/{plan_year}"
    os.makedirs(year_folder, exist_ok=True)
    zip_path = f"{year_folder}/{source_url.rsplit('/', 1)[-1]}"

    # Hash while downloading, so the fingerprint is of exactly the bytes CMS sent.
    sha256 = hashlib.sha256()
    total_bytes = 0
    with requests.get(source_url, headers=REQUEST_HEADERS, stream=True, timeout=300) as response:
        response.raise_for_status()
        with open(zip_path, "wb") as out_file:
            for chunk in response.iter_content(chunk_size=CHUNK_BYTES):
                out_file.write(chunk)
                sha256.update(chunk)
                total_bytes += len(chunk)

    with zipfile.ZipFile(zip_path) as archive:
        contents = [f"{m.filename} ({m.file_size:,} bytes)" for m in archive.infolist() if not m.is_dir()]

    return {
        "plan_year": plan_year,
        "dataset": dataset,
        "source_url": source_url,
        "saved_to": zip_path,
        "zip_bytes": total_bytes,
        "zip_sha256": sha256.hexdigest(),
        "zip_contents": " | ".join(contents),
    }


results = []
for plan_year, dataset, source_url in CMS_SOURCE_FILES:
    print(f"Downloading {plan_year} {dataset} ...")
    try:
        results.append(download_and_fingerprint(plan_year, dataset, source_url))
        print("   done")
    except Exception as error:  # keep going so one bad link doesn't stop the others
        print(f"   FAILED: {error}")

display(pd.DataFrame(results))

In [0]:
# Unzips each CMS file and records its "receipt" (fingerprints + true row count from the
# original CSV) in source_manifest. verify_data_pull later compares loaded tables to these
# numbers, so they must come from the raw files, not from the Delta tables.

import csv
import hashlib
import zipfile
from datetime import datetime, timezone

import pandas as pd
from pyspark.sql import functions as F

VOLUME_DIR = "/Volumes/<your_catalog>/<your_schema>"
MANIFEST_TABLE = "<your_catalog>.<your_schema>.source_manifest"

# (plan_year, dataset, official CMS download URL, zip name, csv name inside the zip)
CMS_SOURCE_FILES = [
    (2026, "plan_attributes",           "https://download.cms.gov/marketplace-puf/2026/plan-attributes-puf.zip",           "plan-attributes-puf.zip",           "plan-attributes-puf.csv"),
    (2026, "rates",                     "https://download.cms.gov/marketplace-puf/2026/rate-puf.zip",                      "rate-puf.zip",                      "rate-puf.csv"),
    (2026, "benefits_and_cost_sharing", "https://download.cms.gov/marketplace-puf/2026/benefits-and-cost-sharing-puf.zip", "benefits-and-cost-sharing-puf.zip", "benefits-and-cost-sharing-puf.csv"),
    (2027, "plan_attributes",           "https://download.cms.gov/marketplace-puf/2027/plan-attributes-puf.zip",           "plan-attributes-puf.zip",           "plan-attributes-puf.csv"),
    (2027, "rates",                     "https://download.cms.gov/marketplace-puf/2027/rate-puf.zip",                      "rate-puf.zip",                      "rate-puf.csv"),
    (2027, "benefits_and_cost_sharing", "https://download.cms.gov/marketplace-puf/2027/benefits-and-cost-sharing-puf.zip", "benefits-and-cost-sharing-puf.zip", "benefits-and-cost-sharing-puf.csv"),
]

csv.field_size_limit(2**31 - 1)  # some CMS text fields are long; avoid false parse errors
CHUNK_BYTES = 8 * 1024 * 1024


def sha256_of_file(path: str) -> tuple[str, int]:
    digest, total = hashlib.sha256(), 0
    with open(path, "rb") as f:
        while chunk := f.read(CHUNK_BYTES):
            digest.update(chunk)
            total += len(chunk)
    return digest.hexdigest(), total


def count_csv_data_rows(path: str) -> tuple[int, int]:
    """Counts CSV records (not text lines) so quoted fields with line breaks count once."""
    with open(path, newline="", encoding="utf-8-sig", errors="replace") as f:
        reader = csv.reader(f)
        header = next(reader)
        data_rows = sum(1 for row in reader if row)  # skip fully blank trailing lines
    return data_rows, len(header)


manifest_rows = []
for plan_year, dataset, source_url, zip_name, csv_name in CMS_SOURCE_FILES:
    print(f"Processing {plan_year} {dataset} ...")
    try:
        zip_path = f"{VOLUME_DIR}/{plan_year}/{zip_name}"
        extract_dir = f"{VOLUME_DIR}/{plan_year}/extracted"
        with zipfile.ZipFile(zip_path) as archive:
            archive.extract(csv_name, path=extract_dir)
        csv_path = f"{extract_dir}/{csv_name}"

        zip_sha256, _ = sha256_of_file(zip_path)
        csv_sha256, csv_bytes = sha256_of_file(csv_path)
        data_rows, column_count = count_csv_data_rows(csv_path)

        manifest_rows.append({
            "dataset": dataset,
            "plan_year": plan_year,
            "source_url": source_url,
            "file_sha256": zip_sha256,
            "row_count": data_rows,
            "csv_file": csv_path,
            "csv_sha256": csv_sha256,
            "file_bytes": csv_bytes,
            "column_count": column_count,
            "recorded_at": datetime.now(timezone.utc).isoformat(),
        })
        print(f"   done: {data_rows:,} data rows, {column_count} columns")
    except Exception as error:
        print(f"   FAILED: {error}")

if len(manifest_rows) == len(CMS_SOURCE_FILES):
    (spark.createDataFrame(pd.DataFrame(manifest_rows))
        .withColumn("plan_year", F.col("plan_year").cast("int"))
        .write.mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(MANIFEST_TABLE))
    print(f"Saved {len(manifest_rows)} rows to {MANIFEST_TABLE}")
    display(spark.table(MANIFEST_TABLE).orderBy("dataset", "plan_year"))
else:
    print("Manifest NOT saved because at least one file failed. Paste the FAILED line(s) to me.")